In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install huggingface_hub

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Set the environment variable so huggingface_hub automatically uses it
import os
os.environ["HF_TOKEN"] = hf_token

# Verify the login (optional)
!hf auth whoami

In [ ]:
# @title 1. 環境構築とGemmaモデルの準備
# 必要なライブラリのインストール
!pip install -q torch torchvision transformers accelerate bitsandbytes

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
from threading import Thread
import sys
import io
import time

# --- 設定 ---
# GPUでも動く軽量なモデルを選択 (gemma-2-2b-it推奨)
MODEL_ID = "google/gemma-2-2b-it" 

print(f"🔄 {MODEL_ID} をロード中... (数分かかります)")

# トークナイザーとモデルのロード
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

print("✅ モデルロード完了。AIティーチャーの準備ができました。")

# --- 思考プロセス表示用の関数 ---
def stream_thought(prompt, max_new_tokens=512):
    """
    LLMに思考過程を語らせ、リアルタイムで出力する関数
    """
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    
    generation_kwargs = dict(
        inputs, 
        streamer=streamer, 
        max_new_tokens=max_new_tokens,
        temperature=0.7
    )
    
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()
    
    print("\n🤖 [AI思考ログ] -----------------------------------------")
    generated_text = ""
    for new_text in streamer:
        sys.stdout.write(new_text)
        sys.stdout.flush()
        generated_text += new_text
    print("\n--------------------------------------------------------\n")
    return generated_text

# --- 実行ログのフック (擬似的な物理/低レイヤーログ) ---
class VerboseNetwork(nn.Module):
    def __init__(self):
        super(VerboseNetwork, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28*28, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)
        
    def forward(self, x):
        # データが流れる様子を詳細に出力
        print(f"   [Hardware Info] Input Tensor Shape: {x.shape} | Device: {x.device}")
        
        x = self.flatten(x)
        print(f"   [Layer Op] Flatten -> Shape: {x.shape}")
        
        x = self.fc1(x)
        print(f"   [Matrix Mul] FC1 (784 -> 128) | Active Neurons: {(x > 0).sum().item()}")
        
        x = self.relu(x)
        print(f"   [Activation] ReLU Applied.")
        
        x = self.fc2(x)
        print(f"   [Matrix Mul] FC2 (128 -> 10) | Output Raw Scores Calculated")
        return x

In [ ]:
# @title 2. リアルタイム解説付きMNIST学習実行
# データセットの準備
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

# ネットワークとオプティマイザ
net = VerboseNetwork().to("cuda")
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.01)

# --- AIによる解説プロンプトの構成 ---
def create_teaching_prompt(step_name, technical_details):
    return f"""
    あなたはコンピュータサイエンスの先生です。
    今、以下の処理を行おうとしています: 「{step_name}」
    
    技術的な詳細:
    {technical_details}
    
    この処理がコンピュータの中で物理的に、また論理的にどう行われているか、
    初学者にもわかるように、しかし専門用語（行列演算、勾配降下法、VRAM転送など）を交えて
    リアルタイムの実況風に解説してください。
    """

# --- 学習ループ (解説付き) ---

# 1. データロードの解説
stream_thought(create_teaching_prompt(
    "データのGPU転送", 
    "Batch Size: 64, Image Size: 28x28. CPU RAMからGPU VRAMへTensorを転送します。"
))

# デモ用に1バッチだけ実行して詳細を見せる
data_iter = iter(train_loader)
images, labels = next(data_iter)
images, labels = images.to("cuda"), labels.to("cuda")

# 2. 順伝播 (Forward) の解説
stream_thought(create_teaching_prompt(
    "ニューラルネットワークの順伝播", 
    "入力層から隠れ層、出力層へと信号が伝わります。行列積(Matrix Multiplication)が行われます。"
))

print("⚡ [System Log] 順伝播プロセス開始...")
outputs = net(images)
print("✅ [System Log] 順伝播完了")

# 3. 損失計算と逆伝播の解説
loss = criterion(outputs, labels)
stream_thought(create_teaching_prompt(
    "誤差逆伝播 (Backpropagation)", 
    f"計算されたLoss: {loss.item():.4f}。この誤差を元に、各パラメータの勾配(Gradient)を計算します。"
))

optimizer.zero_grad()
loss.backward()
print(f"📉 [System Log] Backward pass complete. Loss: {loss.item():.4f}")

# 4. パラメータ更新の解説
stream_thought(create_teaching_prompt(
    "重みの更新 (Optimizer Step)", 
    "SGD (確率的勾配降下法) を使用して、計算された勾配の逆方向に重みをわずかに移動させます。"
))

optimizer.step()
print("🔄 [System Log] Weights updated.")

stream_thought("最後に、ここまでの1ステップで何が行われたか、生徒に向けて一言で励ましのメッセージをください。")

In [ ]:
# @title 1. TPU環境構築とライブラリのインストール
# Keras 3 と KerasNLP をインストール（JAXバックエンド用）
!pip install -q -U keras-nlp
!pip install -q -U keras>=3

import os

# JAXをバックエンドに指定（TPUの性能を最大化するため）
os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.00" # メモリをフル活用

import keras
import keras_nlp
import jax
import numpy as np

# --- TPUの検出と初期化 ---
print("🚀 TPU初期化プロセス開始...")
try:
    # TPUデバイスの確認
    tpu = jax.devices()
    print(f"✅ TPU検出成功: {len(tpu)} コアが利用可能です。")
    print(f"   デバイス詳細: {tpu}")
except:
    print("⚠️ TPUが見つかりません。ランタイムの設定を確認してください。CPU/GPUで動作します。")

# 混合精度演算の設定（計算速度向上とメモリ節約）
keras.mixed_precision.set_global_policy("mixed_bfloat16")
print("⚡ Mixed Precision (bfloat16) を有効化しました。")

In [ ]:
# @title 2. Gemmaモデルのロードと「思考する学習ループ」の準備
# 軽量な2Bモデルを使用 (colab TPUで余裕を持って動くサイズ)
MODEL_ID = "gemma2_2b_en" 

print(f"\n📥 {MODEL_ID} をロード中... (JAX用にコンパイルされます)")
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset(MODEL_ID)
print("✅ モデルロード完了")

# --- 詳細実況のためのカスタムコールバック ---
class ThoughtProcessLogger(keras.callbacks.Callback):
    """
    学習の1ステップごとに、TPU内部の挙動や思考過程（Lossの変化）を
    詳細に実況するコールバック
    """
    def on_train_begin(self, logs=None):
        print("\n🤖 [AI Teacher] ファインチューニングを開始します。")
        print("   ここからの目標は、汎用的なGemmaモデルを、特定のデータセットに特化させることです。")
        print("   TPUの全コア(8コア)を使って、データ並列処理で勾配を計算します。\n")

    def on_epoch_begin(self, epoch, logs=None):
        print(f"📅 [Epoch {epoch + 1}] 開始")
        print("   データをTPUメモリに分散転送中...")

    def on_train_batch_end(self, batch, logs=None):
        # 5バッチごとに詳細な思考ログを出力
        if batch % 5 == 0:
            loss = logs['loss']
            # 擬似的な内部状態の解説
            print(f"\n   🔍 [Step {batch}] ----------------------------------------")
            print(f"   📉 Current Loss: {loss:.4f}")
            
            if loss > 2.0:
                print("      👉 まだ誤差が大きいです。モデルは入力と出力の関係を模索中。")
                print("      👉 Backward Pass: 勾配が大きく変動しており、重みが大きく更新されています。")
            elif loss > 1.0:
                print("      👉 誤差が縮まってきました。文法やパターンを掴み始めています。")
                print("      👉 Optimizer: 学習率に従い、パラメータの微調整フェーズに入りつつあります。")
            else:
                print("      👉 非常に低いLossです！モデルはデータセットの特徴をほぼ完全に捉えました。")
            
            print("      💾 [Hardware] TPU Matrix Units (MXU) Utilization: High")
            print("   --------------------------------------------------------")

# --- LoRA (Low-Rank Adaptation) の設定 ---
# 全パラメータを学習すると重すぎるため、LoRAで効率化します
print("\n🔧 LoRA (Low-Rank Adaptation) を適用中...")
print("   説明: 巨大な行列を直接更新せず、低ランク行列の積として近似更新します。")
print("   効果: 学習可能なパラメータ数を劇的に（1/100以下に）削減します。")

gemma_lm.backbone.enable_lora(rank=4)
gemma_lm.preprocessor.sequence_length = 512 # シーケンス長

# 学習対象のパラメータ数を表示
gemma_lm.summary()

In [ ]:
# @title 3. データセット準備とファインチューニング実行
# デモ用のデータセット（JSON形式などで本来は用意するが、ここではリストで作成）
# 例えば「AIアシスタントとしての振る舞い」を教えるデータ
data = [
    "User: Hello, who are you? \nModel: I am Gemma, an AI assistant developed by Google.",
    "User: What is TPU? \nModel: TPU stands for Tensor Processing Unit, an AI accelerator application-specific integrated circuit (ASIC).",
    "User: Explain normalization. \nModel: Normalization is a technique to scale input data to a specific range, often improving convergence speed.",
    "User: Python code for loop. \nModel: for i in range(10): print(i)",
    # データを複製してバッチ数を稼ぐ（デモ用）
] * 20 

print(f"\n📚 学習データ: {len(data)} 件のサンプルを準備しました。")

# オプティマイザの設定 (AdamW)
# JAX環境ではコンパイル時に最適化されるため高速です
optimizer = keras.optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.01,
)

gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=["accuracy"],
)

print("\n🚀 ファインチューニングを実行します（実況ログ付き）...")

# 学習開始
# ThoughtProcessLoggerにより、学習経過がリアルタイムで「解説」されます
gemma_lm.fit(
    data, 
    epochs=1, 
    batch_size=4,
    callbacks=[ThoughtProcessLogger()]
)

print("\n✅ 学習完了。LoRAウェイトが更新されました。")

# --- 推論テスト ---
print("\n🧪 [Inference Test] 学習後のモデルで生成テスト:")
prompt = "User: What is TPU? \nModel:"
print(f"Input: {prompt}")

# 生成
generated = gemma_lm.generate(prompt, max_length=64)
print(f"Output: {generated}")

In [ ]:
# @title 1. TPU環境構築とライブラリのインストール
# Keras 3 と KerasNLP をインストール（JAXバックエンド用）
!pip install -q -U keras-nlp
!pip install -q -U keras>=3

import os

# JAXをバックエンドに指定（TPUの性能を最大化するため）
os.environ["KERAS_BACKEND"] = "jax"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.00" # メモリをフル活用

import keras
import keras_nlp
import jax
import numpy as np

# --- TPUの検出と初期化 ---
print("🚀 TPU初期化プロセス開始...")
try:
    # TPUデバイスの確認
    tpu = jax.devices()
    print(f"✅ TPU検出成功: {len(tpu)} コアが利用可能です。")
    print(f"   デバイス詳細: {tpu}")
except:
    print("⚠️ TPUが見つかりません。ランタイムの設定を確認してください。CPU/GPUで動作します。")

# 混合精度演算の設定（計算速度向上とメモリ節約）
keras.mixed_precision.set_global_policy("mixed_bfloat16")
print("⚡ Mixed Precision (bfloat16) を有効化しました。")

In [ ]:
# @title 2. Gemmaモデルのロードと「思考する学習ループ」の準備
# 軽量な2Bモデルを使用 (colab TPUで余裕を持って動くサイズ)
MODEL_ID = "gemma2_2b_en" 

print(f"\n📥 {MODEL_ID} をロード中... (JAX用にコンパイルされます)")
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset(MODEL_ID)
print("✅ モデルロード完了")

# --- 詳細実況のためのカスタムコールバック ---
class ThoughtProcessLogger(keras.callbacks.Callback):
    """
    学習の1ステップごとに、TPU内部の挙動や思考過程（Lossの変化）を
    詳細に実況するコールバック
    """
    def on_train_begin(self, logs=None):
        print("\n🤖 [AI Teacher] ファインチューニングを開始します。")
        print("   ここからの目標は、汎用的なGemmaモデルを、特定のデータセットに特化させることです。")
        print("   TPUの全コア(8コア)を使って、データ並列処理で勾配を計算します。\n")

    def on_epoch_begin(self, epoch, logs=None):
        print(f"📅 [Epoch {epoch + 1}] 開始")
        print("   データをTPUメモリに分散転送中...")

    def on_train_batch_end(self, batch, logs=None):
        # 5バッチごとに詳細な思考ログを出力
        if batch % 5 == 0:
            loss = logs['loss']
            # 擬似的な内部状態の解説
            print(f"\n   🔍 [Step {batch}] ----------------------------------------")
            print(f"   📉 Current Loss: {loss:.4f}")
            
            if loss > 2.0:
                print("      👉 まだ誤差が大きいです。モデルは入力と出力の関係を模索中。")
                print("      👉 Backward Pass: 勾配が大きく変動しており、重みが大きく更新されています。")
            elif loss > 1.0:
                print("      👉 誤差が縮まってきました。文法やパターンを掴み始めています。")
                print("      👉 Optimizer: 学習率に従い、パラメータの微調整フェーズに入りつつあります。")
            else:
                print("      👉 非常に低いLossです！モデルはデータセットの特徴をほぼ完全に捉えました。")
            
            print("      💾 [Hardware] TPU Matrix Units (MXU) Utilization: High")
            print("   --------------------------------------------------------")

# --- LoRA (Low-Rank Adaptation) の設定 ---
# 全パラメータを学習すると重すぎるため、LoRAで効率化します
print("\n🔧 LoRA (Low-Rank Adaptation) を適用中...")
print("   説明: 巨大な行列を直接更新せず、低ランク行列の積として近似更新します。")
print("   効果: 学習可能なパラメータ数を劇的に（1/100以下に）削減します。")

gemma_lm.backbone.enable_lora(rank=4)
gemma_lm.preprocessor.sequence_length = 512 # シーケンス長

# 学習対象のパラメータ数を表示
gemma_lm.summary()

In [ ]:
# @title 3. データセット準備とファインチューニング実行
# デモ用のデータセット（JSON形式などで本来は用意するが、ここではリストで作成）
# 例えば「AIアシスタントとしての振る舞い」を教えるデータ
data = [
    "User: Hello, who are you? \nModel: I am Gemma, an AI assistant developed by Google.",
    "User: What is TPU? \nModel: TPU stands for Tensor Processing Unit, an AI accelerator application-specific integrated circuit (ASIC).",
    "User: Explain normalization. \nModel: Normalization is a technique to scale input data to a specific range, often improving convergence speed.",
    "User: Python code for loop. \nModel: for i in range(10): print(i)",
    # データを複製してバッチ数を稼ぐ（デモ用）
] * 20 

print(f"\n📚 学習データ: {len(data)} 件のサンプルを準備しました。")

# オプティマイザの設定 (AdamW)
# JAX環境ではコンパイル時に最適化されるため高速です
optimizer = keras.optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.01,
)

gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=["accuracy"],
)

print("\n🚀 ファインチューニングを実行します（実況ログ付き）...")

# 学習開始
# ThoughtProcessLoggerにより、学習経過がリアルタイムで「解説」されます
gemma_lm.fit(
    data, 
    epochs=1, 
    batch_size=4,
    callbacks=[ThoughtProcessLogger()]
)

print("\n✅ 学習完了。LoRAウェイトが更新されました。")

# --- 推論テスト ---
print("\n🧪 [Inference Test] 学習後のモデルで生成テスト:")
prompt = "User: What is TPU? \nModel:"
print(f"Input: {prompt}")

# 生成
generated = gemma_lm.generate(prompt, max_length=64)
print(f"Output: {generated}")